# Tech Challenge — Fase 1 | Data Analytics

**Case E-commerce Olist**

Análise construída manualmente com foco em desempenho comercial, eficiência logística e satisfação do cliente.


In [ ]:
#bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Puxando o dataset com o pandas
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')

In [ ]:
# verificando o dataset correto
print(orders.shape)
print(items.shape)
print(reviews.shape)

In [ ]:
#verificando a estrutura das tabelas
print("ORDERS")
print(orders.info())

print("\nITEMS")
print(items.info())

print("\nREVIEWS")
print(reviews.info())

In [ ]:
#visualizar o conteudo das tabelas
orders.head()

In [ ]:
items.head ()

In [ ]:
reviews.head()

In [ ]:
# Inicio do tratamento de dados
# convertendo as datas
colunas_data = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for coluna in colunas_data:
    orders[coluna] = pd.to_datetime(orders[coluna])

In [ ]:
orders.info()

In [ ]:
#Avaliar o status dos pedidos que estão com a coluna de delivery null
orders['order_status'].value_counts()

In [ ]:
#Inconsitencia de 8 registors
orders.isnull().sum()

In [ ]:
#melhorando a vizualização de dados, colunas confundindo
orders = orders.rename(columns={
    'order_id': 'id_pedido',
    'customer_id': 'id_cliente',
    'order_status': 'status_pedido',
    'order_purchase_timestamp': 'data_compra',
    'order_approved_at': 'data_aprovacao',
    'order_delivered_carrier_date': 'data_envio_transportadora',
    'order_delivered_customer_date': 'data_entrega_cliente',
    'order_estimated_delivery_date': 'data_entrega_prevista'
})

In [ ]:
orders[
    (orders['status_pedido'] == 'delivered') &
    (orders['data_entrega_cliente'].isnull())
]

Qualidade dos dados: Foram identificados 8 pedidos classificados como entregues sem registro da data efetiva de entrega. Esses registros serão preservados na base principal, porém desconsiderados nas análises que dependem do cálculo do prazo de entrega.

## Análise de desempenho logístico

Para analisar a eficiência logística, serão considerados apenas pedidos
classificados como entregues e que possuem registro da data efetiva de entrega.

In [ ]:
# Criando base específica para análise logística

orders_logistica = orders[
    (orders['status_pedido'] == 'delivered') &
    (orders['data_entrega_cliente'].notnull())
].copy()

orders_logistica.shape

In [ ]:
# Calculando o tempo entre a compra e a entrega ao cliente

orders_logistica['dias_entrega'] = (
    orders_logistica['data_entrega_cliente'] -
    orders_logistica['data_compra']
).dt.days

In [ ]:
orders_logistica[
    ['data_compra', 'data_entrega_cliente', 'dias_entrega']
].head(10)

In [ ]:
# Calculando diferença entre a entrega real e a entrega prevista

orders_logistica['dias_atraso'] = (
    orders_logistica['data_entrega_cliente'] -
    orders_logistica['data_entrega_prevista']
).dt.days

In [ ]:
orders_logistica[
    [
        'data_entrega_cliente',
        'data_entrega_prevista',
        'dias_atraso'
    ]
].head(10)

In [ ]:
# INICIANDO ESTATÍSTICA!!!
orders_logistica[['dias_entrega', 'dias_atraso']].describe()

#Primeiros insights logísticos

Nos 96.470 pedidos analisados, o tempo médio de entrega foi de 12 dias, com mediana de 10 dias. As entregas ocorreram, em média, 12 dias antes do prazo previsto.

Apesar do bom desempenho geral, foram identificados valores extremos de entrega e atraso, que serão investigados nas próximas análises.

In [ ]:
# Classificar entregas no prazo
orders_logistica['entrega_no_prazo'] = (
    orders_logistica['dias_atraso'] <= 0
)

In [ ]:
# Quantidade
orders_logistica['entrega_no_prazo'].value_counts()

In [ ]:
# Percentual
orders_logistica['entrega_no_prazo'].value_counts(normalize=True) * 100

##KPI — Entregas no prazo

93,23% dos pedidos foram entregues dentro ou antes do prazo previsto, enquanto 6,77% apresentaram atraso. O resultado indica alto cumprimento dos prazos estimados, apesar da existência de casos extremos identificados anteriormente.

In [ ]:
#Primeiro grafico executivo com base no nosso primeiro KPI
import matplotlib.pyplot as plt
import seaborn as sns

percentual_prazo = (
    orders_logistica['entrega_no_prazo']
    .value_counts(normalize=True)
    .mul(100)
    .reset_index()
)

percentual_prazo.columns = ['entrega_no_prazo', 'percentual']

percentual_prazo['status'] = percentual_prazo['entrega_no_prazo'].map({
    True: 'No prazo',
    False: 'Atrasado'
})

plt.figure(figsize=(8, 5))

sns.barplot(
    data=percentual_prazo,
    x='status',
    y='percentual'
)

plt.title('Cumprimento do prazo de entrega')
plt.xlabel('')
plt.ylabel('Percentual de pedidos (%)')
plt.grid(axis='y', alpha=0.3)

for container in plt.gca().containers:
    plt.gca().bar_label(
        container,
        fmt='%.1f%%',
        padding=3
    )
plt.show()

In [ ]:
# Proxima tabela, deixar as colunas amigaveis
reviews = reviews.rename(columns={
    'review_id': 'id_avaliacao',
    'order_id': 'id_pedido',
    'review_score': 'nota_avaliacao',
    'review_comment_title': 'titulo_avaliacao',
    'review_comment_message': 'comentario_avaliacao',
    'review_creation_date': 'data_avaliacao',
    'review_answer_timestamp': 'data_resposta_avaliacao'
})

In [ ]:
reviews.head()

In [ ]:
#iniciando um merge para vincular as tabelas de reviews e orders_logistica
reviews['id_pedido'].duplicated().sum()

In [ ]:
reviews['id_pedido'].nunique()

existe uma pequena relação 1:N entre pedido e avaliação em alguns casos.

In [ ]:
reviews_pedido = (
    reviews.groupby('id_pedido')['nota_avaliacao']
    .mean()
    .reset_index()
)

In [ ]:
reviews_pedido.shape

In [ ]:
analise_logistica_reviews = orders_logistica.merge(
    reviews_pedido,
    on='id_pedido',
    how='inner'
)


Pedidos com múltiplas avaliações foram consolidados em uma única nota média antes do relacionamento entre as bases, evitando duplicidade nas análises.



In [ ]:
analise_logistica_reviews.shape

In [ ]:
media_avaliacao = (
    analise_logistica_reviews
    .groupby('entrega_no_prazo')['nota_avaliacao']
    .mean()
)

media_avaliacao

##Impacto do atraso na satisfação

Pedidos entregues no prazo apresentaram avaliação média de 4,29, enquanto pedidos atrasados tiveram média de apenas 2,27. Os dados indicam uma forte associação entre o cumprimento do prazo e a satisfação do cliente.

In [ ]:
#Construção do segundo grafico para consolidar notas de satisfação com base na entrega
media_avaliacao = (
    analise_logistica_reviews
    .groupby('entrega_no_prazo')['nota_avaliacao']
    .mean()
    .reset_index()
)

media_avaliacao['status'] = media_avaliacao['entrega_no_prazo'].map({
    True: 'No prazo',
    False: 'Atrasado'
})

plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=media_avaliacao,
    x='status',
    y='nota_avaliacao'
)

plt.title('Avaliação média por cumprimento do prazo')
plt.xlabel('')
plt.ylabel('Avaliação média')
plt.ylim(0, 5)

for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3)

plt.show()

In [ ]:
# Criando base apenas com pedidos atrasados
pedidos_atrasados = analise_logistica_reviews[
    analise_logistica_reviews['dias_atraso'] > 0
].copy()

In [ ]:
# Criando faixas de atraso
pedidos_atrasados['faixa_atraso'] = pd.cut(
    pedidos_atrasados['dias_atraso'],
    bins=[0, 5, 10, 20, float('inf')],
    labels=[
        '1 a 5 dias',
        '6 a 10 dias',
        '11 a 20 dias',
        'Mais de 20 dias'
    ]
)

In [ ]:
avaliacao_por_atraso = (
    pedidos_atrasados
    .groupby('faixa_atraso', observed=True)['nota_avaliacao']
    .mean()
    .reset_index()
)

avaliacao_por_atraso

####Impacto da duração do atraso

Pedidos com atraso de até 5 dias apresentam nota média de 2,99. A partir de 6 dias de atraso, a avaliação cai para aproximadamente 1,7, permanecendo em patamar baixo mesmo em atrasos maiores.

In [ ]:
#grafico da evolução da piora da satisfação com base no atraso

plt.figure(figsize=(9, 5))

ax = sns.barplot(
    data=avaliacao_por_atraso,
    x='faixa_atraso',
    y='nota_avaliacao'
)

plt.title('Avaliação média por faixa de atraso')
plt.xlabel('Faixa de atraso')
plt.ylabel('Avaliação média')
plt.ylim(0, 5)

for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', padding=3)

plt.show()

###Insight — Atraso e satisfação

A avaliação cai de 2,99 nos atrasos de até 5 dias para cerca de 1,7 quando o atraso supera 5 dias, indicando maior impacto na satisfação a partir desse período.

In [ ]:
#tornando a prox tabela de itens amigavel


items = items.rename(columns={
    'order_id': 'id_pedido',
    'order_item_id': 'id_item_pedido',
    'product_id': 'id_produto',
    'seller_id': 'id_vendedor',
    'shipping_limit_date': 'data_limite_envio',
    'price': 'preco',
    'freight_value': 'valor_frete'
})

items['data_limite_envio'] = pd.to_datetime(
    items['data_limite_envio']
)

In [ ]:
items.info()


In [ ]:
# Valor bruto dos produtos vendidos (GMV)
gmv_total = items['preco'].sum()

gmv_total

In [ ]:
quantidade_itens = len(items)
quantidade_pedidos = items['id_pedido'].nunique()

print("Itens vendidos:", quantidade_itens)
print("Pedidos únicos:", quantidade_pedidos)

In [ ]:
ticket_medio = gmv_total / quantidade_pedidos

print(f"Ticket médio: R$ {ticket_medio:.2f}")

###Indicadores comerciais

A base registra aproximadamente RS 13,6 milhões em GMV, distribuídos em 98.666 pedidos e 112.650 itens vendidos, com ticket médio de  RS 137,75 por pedido.

In [ ]:
# Consolidando o valor dos itens por pedido

valor_por_pedido = (
    items.groupby('id_pedido')['preco']
    .sum()
    .reset_index()
)

valor_por_pedido.head()

In [ ]:
comercial = valor_por_pedido.merge(
    orders[['id_pedido', 'data_compra', 'status_pedido']],
    on='id_pedido',
    how='inner'
)

In [ ]:
comercial.groupby('status_pedido')['preco'].agg(
    pedidos='count',
    gmv='sum'
).sort_values('gmv', ascending=False)

In [ ]:
# Base comercial considerando vendas efetivamente entregues

comercial_entregue = comercial[
    comercial['status_pedido'] == 'delivered'
].copy()

In [ ]:
gmv_entregue = comercial_entregue['preco'].sum()
pedidos_entregues = comercial_entregue['id_pedido'].nunique()
ticket_medio_entregue = gmv_entregue / pedidos_entregues

print(f"GMV entregue: R$ {gmv_entregue:,.2f}")
print(f"Pedidos entregues: {pedidos_entregues}")
print(f"Ticket médio: R$ {ticket_medio_entregue:.2f}")

Desempenho comercial

Os pedidos efetivamente entregues movimentaram RS 13,22 milhões, distribuídos em 96.478 pedidos, com ticket médio de RS 137,04.

In [ ]:
# Criando período mensal da compra
comercial_entregue['mes_compra'] = (
    comercial_entregue['data_compra']
    .dt.to_period('M')
)

In [ ]:
# Evolução mensal do desempenho comercial
evolucao_mensal = (
    comercial_entregue
    .groupby('mes_compra')
    .agg(
        gmv=('preco', 'sum'),
        pedidos=('id_pedido', 'nunique')
    )
    .reset_index()
)

evolucao_mensal

In [ ]:
# Evolução mensal do GMV e quantidade de pedidos
evolucao_mensal = (
    comercial_entregue
    .groupby('mes_compra')
    .agg(
        gmv=('preco', 'sum'),
        pedidos=('id_pedido', 'nunique')
    )
    .reset_index()
)

evolucao_mensal

In [ ]:
#filtrar de 2017 pra frente
evolucao_grafico = evolucao_mensal[
    evolucao_mensal['mes_compra'] >= pd.Period('2017-01')
].copy()

evolucao_grafico['mes_compra'] = (
    evolucao_grafico['mes_compra'].astype(str)
)

In [ ]:
#grafico de evolução das vendas!

plt.figure(figsize=(12, 5))

sns.lineplot(
    data=evolucao_grafico,
    x='mes_compra',
    y='pedidos',
    marker='o'
)

plt.title('Evolução mensal da quantidade de pedidos')
plt.xlabel('Mês')
plt.ylabel('Quantidade de pedidos')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)

plt.show()

#Evolução comercial

O GMV apresentou forte crescimento ao longo de 2017, alcançando cerca de R$ 988 mil em novembro. Em 2018, a operação permaneceu em um patamar elevado, porém com maior estabilidade e sinais de desaceleração do crescimento.

O crescimento foi sustentado por escala ou aumento de ticket?

In [ ]:
# Calculando ticket médio mensal
evolucao_grafico['ticket_medio'] = (
    evolucao_grafico['gmv'] /
    evolucao_grafico['pedidos']
)

evolucao_grafico[
    ['mes_compra', 'gmv', 'pedidos', 'ticket_medio']
]

In [ ]:
evolucao_grafico['ticket_medio'].describe()

In [ ]:
plt.figure(figsize=(12, 5))

sns.lineplot(
    data=evolucao_grafico,
    x='mes_compra',
    y='ticket_medio',
    marker='o'
)

plt.title('Evolução mensal do ticket médio')
plt.xlabel('Mês')
plt.ylabel('Ticket médio (R$)')
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.ylim(100, 180)

plt.show()

In [ ]:
#INICIANDO PARTES MAISSS ESTATÍSTICAS
plt.figure(figsize=(10, 5))

sns.boxplot(
    data=orders_logistica,
    x='dias_entrega'
)

plt.title('Distribuição do tempo de entrega')
plt.xlabel('Dias para entrega')
plt.grid(axis='x', alpha=0.3)

plt.show()

In [ ]:
Q1 = orders_logistica['dias_entrega'].quantile(0.25)
Q3 = orders_logistica['dias_entrega'].quantile(0.75)

IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Limite inferior:", limite_inferior)
print("Limite superior:", limite_superior)

In [ ]:
# Identificando os potenciais outliers
outliers_entrega = orders_logistica[
    orders_logistica['dias_entrega'] > limite_superior
]

print("Quantidade de outliers:", len(outliers_entrega))

percentual_outliers = (
    len(outliers_entrega) /
    len(orders_logistica)
) * 100

print(f"Percentual de outliers: {percentual_outliers:.2f}%")

In [ ]:
outliers_entrega[
    ['id_pedido', 'dias_entrega', 'dias_atraso']
].sort_values(
    'dias_entrega',
    ascending=False
).head(10)

In [ ]:
# Base sem os potenciais outliers
entregas_sem_outliers = orders_logistica[
    orders_logistica['dias_entrega'] <= limite_superior
]

# Comparação
print("COM OUTLIERS")
print(f"Média: {orders_logistica['dias_entrega'].mean():.2f} dias")
print(f"Mediana: {orders_logistica['dias_entrega'].median():.2f} dias")

print("\nSEM OUTLIERS")
print(f"Média: {entregas_sem_outliers['dias_entrega'].mean():.2f} dias")
print(f"Mediana: {entregas_sem_outliers['dias_entrega'].median():.2f} dias")

In [ ]:
#Iniciar uma analise por região
# Carregando base de clientes
customers = pd.read_csv('olist_customers_dataset.csv')

In [ ]:
# Verificando estrutura da base
customers.info()

In [ ]:
print("Registros:", len(customers))
print("Clientes únicos:", customers['customer_id'].nunique())
print("Estados:", customers['customer_state'].nunique())

In [ ]:
# Renomeando as colunas da base de clientes
customers = customers.rename(columns={
    'customer_id': 'id_cliente',
    'customer_unique_id': 'id_cliente_unico',
    'customer_zip_code_prefix': 'cep_prefixo',
    'customer_city': 'cidade',
    'customer_state': 'estado'
})

In [ ]:
# Adicionando localização à base logística
analise_regional = orders_logistica.merge(
    customers[['id_cliente', 'estado', 'cidade']],
    on='id_cliente',
    how='inner'
)

In [ ]:
analise_regional.shape

In [ ]:
pedidos_estado = (
    analise_regional
    .groupby('estado')
    .agg(
        pedidos=('id_pedido', 'nunique')
    )
    .sort_values('pedidos', ascending=False)
    .reset_index()
)

pedidos_estado.head(10)

In [ ]:
desempenho_estado = (
    analise_regional
    .groupby('estado')
    .agg(
        pedidos=('id_pedido', 'nunique'),
        percentual_no_prazo=('entrega_no_prazo', 'mean'),
        tempo_medio_entrega=('dias_entrega', 'mean')
    )
    .reset_index()
)

# Convertendo proporção para percentual
desempenho_estado['percentual_no_prazo'] *= 100

# Criando percentual de atraso
desempenho_estado['percentual_atraso'] = (
    100 - desempenho_estado['percentual_no_prazo']
)

In [ ]:
estados_relevantes = (
    desempenho_estado[
        desempenho_estado['pedidos'] >= 500
    ]
    .sort_values('percentual_atraso', ascending=False)
)

estados_relevantes[
    [
        'estado',
        'pedidos',
        'percentual_atraso',
        'tempo_medio_entrega'
    ]
]

Desempenho logístico regional

A eficiência logística apresenta diferenças relevantes entre os estados. Enquanto SP, MG e PR registram taxas de atraso próximas ou inferiores a 5%, estados como MA, CE, BA e RJ apresentam índices superiores a 12%. O RJ merece atenção especial pelo elevado volume operacional, com 12.350 pedidos e taxa de atraso de 12,1%.

In [ ]:
analise_logistica_reviews.columns

In [ ]:
reviews_regional = analise_logistica_reviews.merge(
    customers[['id_cliente', 'estado']],
    on='id_cliente',
    how='inner'
)

In [ ]:
satisfacao_estado = (
    reviews_regional
    .groupby('estado')
    .agg(
        pedidos=('id_pedido', 'nunique'),
        nota_media=('nota_avaliacao', 'mean')
    )
    .reset_index()
)

satisfacao_estado = (
    satisfacao_estado[
        satisfacao_estado['pedidos'] >= 500
    ]
    .sort_values('nota_media')
)

satisfacao_estado

Logística e satisfação por região

A análise regional reforça a relação observada entre desempenho logístico e satisfação. Estados com maiores taxas de atraso, como MA, CE, BA e RJ, também apresentam avaliações médias inferiores. O RJ se destaca como ponto de atenção devido ao alto volume de pedidos combinado com taxa de atraso de 12,1% e nota média de 3,97.

# Conclusão Executiva

A análise dos dados da Olist indica uma operação com forte ganho de escala ao longo do período analisado, acompanhada por bom desempenho logístico no agregado e elevada satisfação quando os prazos de entrega são cumpridos.

No aspecto comercial, os pedidos entregues movimentaram aproximadamente RS 13,22 milhões em GMV, com 96.478 pedidos e ticket médio de RS 137,04. O crescimento observado foi impulsionado principalmente pelo aumento do volume de pedidos, enquanto o ticket médio permaneceu relativamente estável.

Na logística, 93,23% das entregas ocorreram dentro do prazo. Entretanto, foram identificados casos extremos e diferenças regionais relevantes. A análise mostrou também uma forte associação entre cumprimento do prazo e satisfação: pedidos entregues no prazo apresentaram nota média de 4,29, contra 2,27 nos pedidos atrasados.

Regionalmente, alguns mercados apresentam maior risco operacional. O Rio de Janeiro merece atenção especial por combinar alto volume de pedidos com taxa de atraso de 12,1% e nota média de 3,97.

De forma geral, os dados indicam uma operação capaz de ganhar escala, mas cuja sustentabilidade depende da manutenção da eficiência logística e da redução dos gargalos que afetam diretamente a experiência do cliente.

# Recomendações Estratégicas

### 1. Priorizar regiões de maior impacto operacional
Direcionar esforços de melhoria logística para estados que combinam volume relevante e elevada taxa de atraso, com atenção especial ao Rio de Janeiro, além de mercados como Bahia, Ceará e Maranhão.

### 2. Atuar preventivamente sobre atrasos críticos
Criar mecanismos de acompanhamento para pedidos com risco de atraso, principalmente aqueles que possam ultrapassar cinco dias além do prazo, faixa em que foi observada forte deterioração da avaliação dos clientes.

### 3. Sustentar o crescimento com eficiência logística
Como o crescimento comercial foi impulsionado principalmente pelo aumento do volume de pedidos, a expansão da operação deve ser acompanhada pelo fortalecimento da capacidade logística, evitando que o ganho de escala comprometa a satisfação do cliente.